Source data is loaded from the project's DBRepo instance via the REST API,
through `src.dbrepo_loader.load_view`. The experiment uses the
`v_measurements_enriched` view, which is the de-normalised join of
`TrafficMeasurements`, `Calendar`, and `TrafficSites` covering January–June
2022 traffic flow measurements from South Dublin County Council (SDCC).

See `README.md` ("DBRepo API access") for endpoint, auth, and parity-check
documentation.


In [ ]:
!nvidia-smi

In [ ]:
!pip install -r colab.txt

In [1]:
# Imports:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import os
import random

/opt/anaconda3/envs/pred_traffic/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
def enforce_reproducibility(seed=42):
    # 1. Set Python core environment seeds
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    # 2. Set TensorFlow execution seeds
    tf.random.set_seed(seed)

    # 3. Restrict GPU operations to deterministic math algorithms
    # Note: This can sometimes cause a minor performance hit on the T4 GPU
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Execute the configuration setup immediately before processing data
enforce_reproducibility(seed=42)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv('/content/.env')

from src.dbrepo_loader import load_view

df_traffic_flow = load_view("v_measurements_enriched")

print(df_traffic_flow.head())
print(df_traffic_flow.info())


In [ ]:
print(df_traffic_flow.describe())

In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.boxplot(data=df_traffic_flow.filter(like='flow'))
plt.title('Flow Boxplot')
plt.xlabel('Flow')

plt.subplot(1, 3, 2)
sns.boxplot(data=df_traffic_flow.filter(like='cong'))
plt.title('Congestion Boxplot')
plt.xlabel('Congestion')

plt.subplot(1, 3, 3)
sns.boxplot(data=df_traffic_flow.filter(like='dsat'))
plt.title('Disaturation Boxplot')
plt.xlabel('Disaturation')

plt.tight_layout()
plt.show()

In [ ]:
print(f"Unique Sites: {df_traffic_flow['site_id'].nunique()}")
print(f"Count per site: \n {df_traffic_flow['site_id'].value_counts()}")


Print visualisations of the distribution of the variables:

In [ ]:
# Set the aesthetic style of the plots
sns.set_style("whitegrid")

# Create a bar plot for the 'site_id' column
plt.figure(figsize=(12, 6))
sns.countplot(data=df_traffic_flow, x='site_id', order=df_traffic_flow['site_id'].value_counts().index)
plt.title('Distribution of Traffic Sites')
plt.xlabel('Site ID')
plt.ylabel('Number of Entries')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.histplot(df_traffic_flow['flow'], bins=50, kde=True)
plt.title('Distribution of Flow')
plt.xlabel('Flow')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.histplot(df_traffic_flow['cong'], bins=50, kde=True)
plt.title('Distribution of Congestion')
plt.xlabel('Congestion')
plt.ylabel('Frequency')

plt.subplot(1, 3, 3)
sns.histplot(df_traffic_flow['dsat'], bins=50, kde=True)
plt.title('Distribution of Disaturation')
plt.xlabel('Disaturation')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

# Classification:
Build a classification model to predict traffic congestion (cong) based on the provided traffic flow data.

Use the prediction categories 1 (free flow) to 5 (severe congestion) and use the quantils per site as classifier.

Split the data into training, testing and validation sets (60-20-20), train the regression model (multi layer perceptron), and evaluate its performance by calculating and visualizing the confusion matrix, precision, recall, and plotting the ROC curve with a legend. Finally, provide a summary of the model's performance.

# Step 1: Preprocessing & Label Creation

Create the target 'cong_class'

| Class | Meaning            |
|-------|--------------------|
| 0     | No congestion (0%) |
| 1     | Low congestion     |
| 2     | Moderate           |
| 3     | High               |
| 4     | Severe             |

**cong:**
Congestion is directly measured from the detector. If the detector is placed beyond the normal end of queue in the street it is rarely covered by stationary traffic, except of course when congestion occurs. If any detector shows standing traffic for the whole of an interval this is recorded. The number of intervals of congestion in any cycle is also recorded.

*The percentage congestion is calculated from:*
'# of congested intervals x 4 x 100 cycle time in seconds.
This percentage of congestion is available to view and more importantly for the optimisers to take into account.

In [ ]:
df_traffic_flow.groupby('site_id')['cong'].head()


In [ ]:
def classify_congestion(group):
    group = group.copy()

    # Initialize result with default class (0 = no congestion)
    result = pd.Series(0, index=group.index, dtype=int)

    # Select non-zero values
    non_zero_idx = group[group > 0].index
    non_zero = group.loc[non_zero_idx]

    # If not enough variation, assign class 1 (low congestion)
    if len(non_zero) < 5 or non_zero.nunique() < 2:
        result.loc[non_zero_idx] = 1
        return result

    # Apply qcut safely
    try:
        quantiles = pd.qcut(
            non_zero,
            4,
            labels=False,
            duplicates='drop'
        ) + 1  # classes 1–4

        # Assign using aligned indices
        result.loc[quantiles.index] = quantiles.astype(int)

    except ValueError:
        # fallback if qcut still fails
        result.loc[non_zero_idx] = 1

    return result

In [ ]:
#def classify_congestion(group):
#    # Separate zero congestion (very common)
#    zero_mask = group == 0
#
#    # For non-zero values, use percentiles
#    non_zero = group[~zero_mask]
#
#    if len(non_zero) < 5:
#        # fallback if too little data
#        return pd.Series([1]*len(group), index=group.index)
#
#    quantiles = pd.qcut(non_zero, 4, labels=False, duplicates='drop') + 2
#    # +2 because class 1 is reserved for zero
#
#    result = pd.Series(index=group.index, dtype=int)
#
#    # Assign classes
#    result[zero_mask] = 1              # no congestion
#    result[~zero_mask] = quantiles    # 2–5
#
#    return result

In [ ]:
df_traffic_flow['target'] = (
    df_traffic_flow
    .groupby('site_id')['cong']
    .transform(classify_congestion)
)


In [ ]:
class_counts = df_traffic_flow['target'].value_counts().sort_index()
class_percent = df_traffic_flow['target'].value_counts(normalize=True).sort_index() * 100

print("Counts:\n", class_counts)
print("\nPercent:\n", class_percent)

In [ ]:
class_counts.plot(kind='bar')
plt.title("Overall Congestion Class Distribution")
plt.xlabel("Congestion Class")
plt.ylabel("Count")
plt.show()

In [ ]:
site_class_dist = (
    df_traffic_flow
    .groupby('site_id')['target']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

site_class_dist.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)

plt.title("Congestion Class Distribution per Site")
plt.xlabel("Site")
plt.ylabel("Proportion")
plt.legend(title="Class")
plt.show()


In [ ]:
top_sites = df_traffic_flow['site_id'].value_counts().head(10).index

subset = site_class_dist.loc[top_sites]

subset.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)

plt.title("Top 10 Sites – Congestion Distribution")
plt.show()


In [ ]:
top_sites = df_traffic_flow['site_id'].value_counts().head(10).index

sns.boxplot(
    data=df_traffic_flow[df_traffic_flow['site_id'].isin(top_sites)],
    x='site_id',
    y='cong'
)

plt.xticks(rotation=45)
plt.title("Congestion Distribution per Site")
plt.show()


In [ ]:
df_traffic_flow['target'].value_counts(normalize=True)

In [ ]:
print(f"max value over all: {df_traffic_flow['cong'].max()}")
print(f"min value over all: {df_traffic_flow['cong'].min()}")
print(f"distribution in 5 bins: \n {df_traffic_flow['cong'].value_counts(bins=5)}")

## 2. Feature Selection & Encoding

We'll use flow, dsat, and 'day_of_week' (encoded) as features.
We ignore Objectid, date, and times for this baseline.


In [ ]:
# 1. Convert start_time to fractional hour (e.g., 14:30 -> 14.5)
df_traffic_flow['hour'] = pd.to_datetime(df_traffic_flow['start_time']).dt.hour
df_traffic_flow['minute'] = pd.to_datetime(df_traffic_flow['start_time']).dt.minute
df_traffic_flow['time_float'] = df_traffic_flow['hour'] + df_traffic_flow['minute'] / 60

In [ ]:
# 2. Cyclical Encoding for Time (Essential for MLPs)
# This teaches the model that 00:00 follows 23:45
df_traffic_flow['sin_time'] = np.sin(2 * np.pi * df_traffic_flow['time_float'] / 24)
df_traffic_flow['cos_time'] = np.cos(2 * np.pi * df_traffic_flow['time_float'] / 24)

In [ ]:
# 3. Map Days to Numbers
day_map = {'MO':0, 'TU':1, 'WE':2, 'TH':3, 'FR':4, 'SA':5, 'SU':6}
df_traffic_flow['day_num'] = df_traffic_flow['day_of_week'].map(day_map)


In [ ]:
# FIX: Fill NaNs (rows that couldn't be binned) with 0 (Class 1)
df_traffic_flow['target'] = df_traffic_flow['target'].fillna(0).astype(int)

# Select Features: Flow and Dsat are the primary indicators
features = ['flow', 'flow_pc', 'dsat', 'dsat_pc', 'sin_time', 'cos_time', 'day_num']
X = df_traffic_flow[features].values
y = df_traffic_flow['target'].values

# Double check features for NaNs as well
X = df_traffic_flow[features].fillna(0).values
y = df_traffic_flow['target'].values

In [ ]:
# Convert to Series and get percentages
distribution = pd.Series(y).value_counts(normalize=True).sort_index() * 100

print("Congestion Class Distribution (%):")
print(distribution)

df_traffic_flow['target'].value_counts(normalize=True)

## Split Data (60-20-20)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
# Convert to Series and get percentages
print(y_train.size)
distribution = pd.Series(y_train).value_counts(normalize=True).sort_index() * 100

print("Congestion Class Distribution (%):")
print(distribution)

# Convert to Series and get percentages
print(y_test.size)
distribution = pd.Series(y_test).value_counts(normalize=True).sort_index() * 100

print("Congestion Class Distribution (%):")
print(distribution)

# Convert to Series and get percentages
print(y_val.size)
distribution = pd.Series(y_val).value_counts(normalize=True).sort_index() * 100

print("Congestion Class Distribution (%):")
print(distribution)

## 4. Scaling (Essential for MLP)

In [ ]:
# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
print(f"count NAN values in flow: {df_traffic_flow['flow'].isna().sum()}")
print(f"count NAN values in flow_pc: {df_traffic_flow['flow_pc'].isna().sum()}")
print(f"count NAN values in dsat: {df_traffic_flow['dsat'].isna().sum()}")
print(f"count NAN values in day_num: {df_traffic_flow['day_num'].isna().sum()}")
print(f"count NAN values in cong: {df_traffic_flow['cong'].isna().sum()}")
print(f"count NAN values in target: {df_traffic_flow['target'].isna().sum()}")

# Step 2: Training the MLP Classifier
We will use a hidden layer architecture (e.g., 64x32) which is a good starting point for this data size.

In [ ]:
# 1. Check for and drop any rows with NaN in features or target
original_count = len(df_traffic_flow)
df_traffic_flow = df_traffic_flow.dropna(subset=features + ['target'])

# 2. Check for Infinite values (rare, but happens with flow/dsat pc)
df_traffic_flow = df_traffic_flow.replace([np.inf, -np.inf], np.nan).dropna(subset=features)

print(f"Dropped {original_count - len(df_traffic_flow)} rows containing NaN or Inf.")

# Now proceed to X, y assignment and scaling
X = df_traffic_flow[features].values.astype('float32') # Force float32 for GPU
y = df_traffic_flow['target'].values.astype('int32')

In [ ]:
print(np.isnan(y_train).sum())

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Calculate weights: Higher weight for smaller classes
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to a dictionary format for Keras
class_weight_dict = dict(enumerate(weights))

print("Calculated Weights:", class_weight_dict)
# You will likely see Class 1 weight ≈ 0.2 and Class 5 weight ≈ 20.0

In [ ]:
# 1. Stop when validation loss hasn't improved for 5 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,               # How many epochs to wait before stopping
    restore_best_weights=True, # Important: Keeps the "best" version of the model
    verbose=1
)

# 2. Lower the learning rate if the model gets stuck
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,               # Multiply learning rate by 0.2
    patience=3,               # Wait 3 epochs before reducing
    min_lr=0.00001,
    verbose=1
)

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)), # New way to define input
    layers.BatchNormalization(),             # Stabilizes the inputs
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(5, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # Explicit learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training with the T4 GPU
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,                  # You can set this higher now; the callback handles stopping
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## Step 3: Evaluation & Visualization
# Confusion Matrix & Metrics

In [2]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

# Predictions
y_probs = model.predict(X_test)
y_pred = np.argmax(y_probs, axis=1)

# Binarize labels for multi-class evaluation
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])

# Calculate the normalized matrix
# 'true' normalizes over the rows (actual labels)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Purples')

plt.title("Normalized Confusion Matrix (Recall per Class)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


NameError: name 'model' is not defined

## Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(10, 8))

for i in range(5):
    # Calculate precision and recall for each class
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_probs[:, i])
    avg_precision = average_precision_score(y_test_bin[:, i], y_probs[:, i])

    plt.plot(recall, precision, label=f'Class {i} (Avg Precision = {avg_precision:0.2f})')

plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision (Positive Predictive Value)')
plt.title('Precision-Recall Curve (Highly Imbalanced Data)')
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.show()

## Multi-class ROC Curve
Since we have 5 classes, we plot a curve for each class using the "One-vs-Rest" approach.

In [ ]:
# ROC Curve
class_names = ['Free Flow', 'Low', 'Medium', 'High', 'Severe']
plt.figure(figsize=(10, 8))
for i in range(5):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {auc(fpr, tpr):.2f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title("ROC Curve (Careful: Can be over-optimistic on imbalanced data)")
plt.legend()
plt.show()